# Caption Filtering and Clip Generation

## Setup

In [1]:
import json, random, re
from pathlib import Path

import pandas as pd
import torch
from huggingface_hub import hf_hub_download

SEED = 7
random.seed(SEED)
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

2.11.0+cu128 True NVIDIA GeForce RTX 5080 Laptop GPU


/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Exploration

In [3]:
DATA_PATH = hf_hub_download("friedrichor/MSR-VTT", "raw_data/MSRVTT_data.json", repo_type="dataset")
DATA = json.load(open(DATA_PATH))

In [4]:
DATA.keys()

dict_keys(['info', 'videos', 'sentences'])

In [5]:
videos = pd.DataFrame(DATA["videos"])[["video_id", "category", "split"]]
captions = pd.DataFrame(DATA["sentences"]).merge(videos, on="video_id")
print(len(videos), "videos,", len(captions), "captions")
captions.head()

10000 videos, 200000 captions


,caption,video_id,sen_id,category,split
0,a cartoon animals runs through an ice cave in ...,video2960,0,2,train
1,a cartoon character runs around inside of a vi...,video2960,1,2,train
2,a character is running in the snow,video2960,2,2,train
3,a person plays a video game centered around ic...,video2960,3,2,train
4,a person plays online and records themselves,video2960,4,2,train


In [6]:
cat_path = hf_hub_download("friedrichor/MSR-VTT", "raw_data/category.txt", repo_type="dataset")
CATEGORIES = {int(i): name for name, i in (l.rsplit("\t", 1) for l in open(cat_path).read().splitlines() if l)}
CATEGORIES

{0: 'music',
 1: 'people',
 2: 'gaming',
 3: 'sports/actions',
 4: 'news/events/politics',
 5: 'education',
 6: 'tv shows',
 7: 'movie/comedy',
 8: 'animation',
 9: 'vehicles/autos',
 10: 'howto',
 11: 'travel',
 12: 'science/technology',
 13: 'animals/pets',
 14: 'kids/family',
 15: 'documentary',
 16: 'food/drink',
 17: 'cooking',
 18: 'beauty/fashion',
 19: 'advertisement'}

In [7]:
pd.set_option("display.max_colwidth", None)
for cat_id, name in CATEGORIES.items():
    sub = captions[captions["category"] == cat_id]
    print(f"\n[{cat_id}] {name}: {sub['video_id'].nunique()} videos, {len(sub)} captions")
    for c in sub["caption"].sample(5, random_state=SEED):
        print("   ", c)


[0] music: 733 videos, 14660 captions
    some girls are dancing
    wrestlers are fighting and showing off in ring
    a woman dressed in black with black hair is dancing at first and then the scene cuts to a blonde vocalist singing and dancing with a group of dancers
    some people sing and dance
    a man in a recording studio is approaching the microphone like he s about to start singing

[1] people: 252 videos, 5040 captions
    there is a woman in white is talking about a product
    a man is showing how to dance
    a man is playing the guitar
    blonde woman in blue dress is talking near the wall
    women smiling and posing for the camera in a bikini with dark brown hair and brown eyes looks to be of asian decent

[2] gaming: 473 videos, 9460 captions
    someone is playing a mario bros
    commentary provided by children for a popular video game called mine craft along with its game play
    minecraft video game walk through with comments
    a guy is playing a game and te

## Filtering the captions

We need to filter out the captions that are not relevant for our analysis, for example, the animations and gaming categories. Other categories may also have irrelevant captions as seen from the earlier samples.

In [20]:
# the four agent groups we study
GROUPS = ("two_people", "one_person", "animal", "vehicle")
TWO_PEOPLE, ONE_PERSON, ANIMAL, VEHICLE = GROUPS

### Prefiltering with spaCy and WordNet

We first group captions cheaply by their agent:
- spaCy finds the active subject of the main verb ("a brown **horse** is playing...").
- WordNet tells us whether that subject is a person, an animal or a vehicle, and the parse tells us how many agents there are.

Names, pronouns, crowds and passive captions ("a car is being driven") are dropped.

In [9]:
import nltk
import spacy
from nltk.corpus import wordnet as wn

nltk.download("wordnet", quiet=True)
nlp = spacy.load("en_core_web_sm", disable=["ner"])

# WordNet roots that define each agent type. agent_type picks by usage counts; a sense under several roots,
# or an exact tie, goes to the first root.
# Animal and vehicle keys are group names, so agent_group can return them as is.
AGENT_ROOTS = {
    ANIMAL:   wn.synset("animal.n.01"),
    VEHICLE:  wn.synset("conveyance.n.03"),   # vehicles and public transport (train, bus)
    "person": wn.synset("person.n.01"),
}

A word can have several meanings in WordNet. We pick the type whose meanings are most common, using WordNet's usage counts. Meanings that are not agents count too, so "ball" (the object) is not a person just because WordNet also lists the actress Lucille Ball. When WordNet has no counts for a word, we trust its first meaning.

In [10]:
def agent_type(token):
    """WordNet type of a noun: the type (or None, for non-agents) whose senses are used most in WordNet's sense counts."""
    
    word = token.lemma_
    senses = wn.synsets(word, pos=wn.NOUN)
    if not senses:
        return None

    weights = [sum(l.count() for l in s.lemmas() if l.name() == word) for s in senses]
    if not any(weights):
        weights = [1] + [0] * (len(senses) - 1) # no usage counts: trust WordNet's first (most common) sense

    scores = dict.fromkeys([*AGENT_ROOTS, None], 0) # None collects non-agent senses ("ball": the object)
    for sense, weight in zip(senses, weights):
        ancestors = set(sense.closure(lambda s: s.hypernyms() + s.instance_hypernyms())) | {sense}
        kind = next((name for name, root in AGENT_ROOTS.items() if root in ancestors), None)
        scores[kind] += weight
    return max(scores, key=scores.get) # first key wins ties

Counting agents: "two men" = 2, "a man and a woman" = 2, a plural without a number ("men") is unknown and gets dropped.

In [11]:
NUMBER_WORDS = {"one": 1, "1": 1, "two": 2, "2": 2, "three": 3, "3": 3}

def agent_count(token):
    """How many agents one subject noun stands for: "two men" -> 2, "a man" -> 1, "men" -> None (unknown)."""
    
    numbers = [NUMBER_WORDS.get(c.text) for c in token.children if c.dep_ == "nummod"]
    
    if numbers:
        return numbers[0]
    return 1 if token.tag_ == "NN" else None

def agent_group(doc):
    """Group a caption by the active subject of its main verb."""
    
    root = next(t for t in doc if t.dep_ == "ROOT")
    if root.pos_ == "VERB": # AUX roots are states ("a pony is in a pen"): dropped
        subjects = [t for t in root.children if t.dep_ == "nsubj"] # active voice only
        if not subjects:
            return None
        head = subjects[0]
    elif root.pos_ == "NOUN" and any(c.dep_ == "acl" and c.pos_ == "VERB" for c in root.children):
        head = root # "a man riding a bike"
    else:
        return None

    agents = [head] + [c for c in head.children if c.dep_ == "conj"] # "a man and a woman"
    if any(a.pos_ != "NOUN" for a in agents):
        return None # names, pronouns ("he", "stephen hawking")
    
    types = {agent_type(a) for a in agents}
    if len(types) != 1 or None in types:
        return None # unknown or mixed agents
    
    kind = types.pop()
    if kind == "person":
        counts = [agent_count(a) for a in agents]
        total = None if None in counts else sum(counts)
        return {1: ONE_PERSON, 2: TWO_PEOPLE}.get(total) # crowds and "some people" -> None
    
    return kind # ANIMAL or VEHICLE

In [12]:
test_samples = ["a man and a woman are talking to each other", 
                "two tigers are fighting",
                "a car is being driven", 
                "two men and a woman sit at a table", 
                "a man riding a bike",
                "a ball is rolling down a hill"]

for sample in test_samples:
    print(sample, "=", agent_group(nlp(sample)))

a man and a woman are talking to each other = two_people
two tigers are fighting = animal
a car is being driven = None
two men and a woman sit at a table = None
a man riding a bike = one_person
a ball is rolling down a hill = None


We also drop the gaming and animation categories, duplicate captions, and captions that are very short or very long.

In [13]:
# drop gaming, animation
prefiltered = captions[~captions["category"].isin({2, 8})].copy()
prefiltered["caption"] = prefiltered["caption"].str.strip().str.lower()
prefiltered = prefiltered.drop_duplicates("caption")
prefiltered = prefiltered[prefiltered["caption"].str.split().str.len().between(5, 15)]

prefiltered["group"] = [agent_group(d) for d in nlp.pipe(prefiltered["caption"], batch_size=1000)]
prefiltered = prefiltered.dropna(subset=["group"])
prefiltered["group"].value_counts()

group
one_person    54571
two_people     5287
animal         2797
vehicle        1637
Name: count, dtype: int64

In [14]:
for group in GROUPS:
    print(f"\n{group}")
    for c in prefiltered[prefiltered["group"] == group]["caption"].sample(5, random_state=SEED):
        print("   ", c)


two_people
    man and woman talking to a camera
    two men dressed as women are on a cooking show
    a husband and wife are greeting each other at the end of a work day
    a man and woman do the whip dance on camera
    a boy and a girl are cuddling and riding bikes

one_person
    a woman talking into a microphone
    a person is getting a tattoo
    a woman wearing a white bikini lays down in snow to make a snow angel
    a person is making some food in a kitchen
    a man in a suit is talking to the camera about calculus

animal
    a male showing how to inject a flavor into the fish on the plate
    cartoon bugs are popping onto a screen
    an animated dog is singing and dancing
    horses running in the derby
    black snake ingests a smaller brown snake

vehicle
    a car is racing around the street
    car moving in tough roads
    a car drives down the road on this game
    a ship is sailing in the water and a beach is shown
    a vehicle is driving around


### Filtering using an LLM

Now we use an LLM to verify that the captions are correctly grouped. We choose Qwen-2.5-7B-Instruct. It was used in one of our references [VBench](https://github.com/Vchitect/VBench)

In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_ID = "Qwen/Qwen2.5-7B-Instruct"

# we use 4-bit weights locally (16GB vram), bf16 on the cluster (24GB vram)
LOAD_4BIT = torch.cuda.get_device_properties(0).total_memory < 20e9

tokenizer = AutoTokenizer.from_pretrained(LLM_ID, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    dtype=torch.bfloat16,
    device_map="cuda",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
    ) if LOAD_4BIT else None,
)

print("4-bit:", LOAD_4BIT, "| VRAM used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

Loading weights: 100%|██████████| 339/339 [00:12<00:00, 28.09it/s]


4-bit: True | VRAM used: 5.9 GB


`chat` runs the LLM on a batch of conversations. Decoding is greedy, so the same input always gives the same output.

In [16]:
@torch.no_grad()
def chat(conversations, max_new_tokens=512, batch_size=8):
    """Greedy-decode a list of chat conversations, batch_size at a time."""
    
    replies = []

    for i in range(0, len(conversations), batch_size):
        batch = conversations[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in batch]
        encode = tokenizer(texts, return_tensors="pt", padding=True).to(llm.device)
        output = llm.generate(**encode, max_new_tokens=max_new_tokens, do_sample=False,
                           temperature=None, top_p=None, top_k=None)
        replies += tokenizer.batch_decode(output[:, encode["input_ids"].shape[1]:], skip_special_tokens=True)
    
    return replies

`parse_json` is taken from OpenBias: keep the text from the first `{` to the last `}`, then parse it.

In [17]:
def parse_json(text):
    """From OpenBias propose_biases(): keep the text from the first { to the last }, then parse."""
    
    if "{" not in text:
        return None

    text = "{" + text.split("{", 1)[1]
    text = text.rsplit("}", 1)[0] + "}"
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

In [18]:
# test the chat function with a simple prompt
print(chat([[{"role": "user", "content": 'Reply with JSON {"ok": true}'}]]))

['{"ok": true}']


The filter prompt instructs the LLM to check whether the caption is relevant for our analysis. It also asks for a reason, which we use to verify that the LLM is not just guessing.

In [19]:
GROUP_DESCRIPTIONS = {
    TWO_PEOPLE: "exactly two people, acting together or on each other",
    ONE_PERSON: "exactly one person",
    ANIMAL:     "one or more animals and no people",
    VEHICLE:    "vehicles and no people or animals",
}
assert set(GROUP_DESCRIPTIONS) == set(GROUPS)
GROUP_LINES = "\n".join(f'- "{g}": {d}' for g, d in GROUP_DESCRIPTIONS.items())

SEED_FILTER_PROMPT = """You select seed captions for a study of text-to-video generators.
Each caption describes a real video. Decide whether the caption is a good seed.

Keep the caption only if ALL of these hold:
1. A real agent acts: one or two people, an animal, or a vehicle. Not a cartoon, game, animation, celebrity or fictional character.
2. The agent does something visible that unfolds over a few seconds. Moving, driving or running counts. Reject static states ("is sitting", "is laying", "is parked", "is on the counter") and one person speaking about a topic ...
3. One continuous shot with one simple event. Reject lists of actions and scene changes.
4. The caption is clear, grammatical English. Reject garbled, misspelled or ambiguous text.
5. A text-to-video model could plausibly render it in a 5-second clip.
Do not reject for content: fights, conflict, alcohol and accidents are valid seeds.

Also give the agent group:
""" + GROUP_LINES + """
- "other": anything else

Answer in JSON only: {"reason": "<10 words>", "group": "...", "keep": true or false}"""

In [20]:
# Few-shot examples, similar to OpenBias style; hand written (not from MSR-VTT)
# Keys in the same order as the prompt's answer format: reason, group, keep
SEED_FILTER_EXAMPLES = [
    ("a horse is trotting across a dirt field",
     {"reason": "real animal, visible motion",
      "group": ANIMAL, "keep": True}),

    ("a truck is speeding down a highway",
     {"reason": "real vehicle, visible motion",
      "group": VEHICLE, "keep": True}),

    ("a woman is chopping vegetables in a kitchen",
     {"reason": "one person, visible action",
      "group": ONE_PERSON, "keep": True}),

    ("a woman is throwing a ball to a man",
     {"reason": "two people, one simple interaction",
      "group": TWO_PEOPLE, "keep": True}),

    ("two dogs are sitting on a couch",
     {"reason": "static state",
      "group": ANIMAL, "keep": False}),

    ("a man is explainng how to fix a bike",
     {"reason": "misspelled; talks about a topic",
      "group": ONE_PERSON, "keep": False}),

    ("a car drives down the road then a crowd cheers",
     {"reason": "two events, scene change",
      "group": VEHICLE, "keep": False}),

    ("a cartoon cat is chasing a mouse",
     {"reason": "cartoon, not a real agent",
      "group": "other", "keep": False}),
]

In [22]:
def seed_filter_messages(caption, examples=SEED_FILTER_EXAMPLES):
    """Chat for one caption: the filter prompt, then each example as a user/assistant turn, then the caption to judge."""
    
    msgs = [{"role": "system", "content": SEED_FILTER_PROMPT}]
    for ex_caption, ex_answer in examples:
        msgs += [{"role": "user", "content": f'Caption: "{ex_caption}"'},
                 {"role": "assistant", "content": json.dumps(ex_answer)}]
    msgs.append({"role": "user", "content": f'Caption: "{caption}"'})
    return msgs

In [23]:
# quick check on a few hand-picked captions
test = ["a man and a woman are talking to each other",
        "two girls are sitting at a counter", # should be rejected, static state
        "a horse is taking bucket which is poink color", # should be rejected, misspelled
        "a dog is chasing a cat around the yard"]

for c, r in zip(test, chat([seed_filter_messages(c) for c in test], max_new_tokens=64)):
    print(c, "->", parse_json(r))

a man and a woman are talking to each other -> {'reason': 'two people, visible interaction', 'group': 'two_people', 'keep': True}
two girls are sitting at a counter -> {'reason': 'static state', 'group': 'two_people', 'keep': False}
a horse is taking bucket which is poink color -> {'reason': 'misspelled, ambiguous', 'group': 'animal', 'keep': False}
a dog is chasing a cat around the yard -> {'reason': 'real animal, visible motion', 'group': 'animal', 'keep': True}


### Running the filter

In [24]:
CANDIDATES_PER_GROUP = 40
SEED_QUOTA = {TWO_PEOPLE: 2, ONE_PERSON: 2, ANIMAL: 4, VEHICLE: 4}

In [25]:
candidates = (prefiltered.drop_duplicates("video_id") # one caption per video
                .groupby("group")
                .sample(n=CANDIDATES_PER_GROUP, random_state=SEED)
                .reset_index(drop=True))

replies = chat([seed_filter_messages(c) for c in candidates["caption"]], max_new_tokens=64)

In [26]:
zip(candidates[:5], replies[:5])
for c, r in zip(candidates[:5]["caption"], replies[:5]):
    print(c, "=", parse_json(r))

in the sea the amazing red seahorse is moving its head = {'reason': 'animal, visible motion', 'group': 'animal', 'keep': True}
camels are grazing in the farm land which has some fences = {'reason': 'visible action, real animals', 'group': 'animal', 'keep': True}
sea creatures move along the ocean bottom = {'reason': 'visible motion, real animals', 'group': 'animal', 'keep': True}
a cartoon monkey is jumping on a pirate ship = {'reason': 'cartoon, not a real agent', 'group': 'other', 'keep': False}
a female fashion show females are walking in and out the lane walkway = {'reason': 'multiple actions, scene change', 'group': 'other', 'keep': False}


In [27]:
judged = [parse_json(r) or {} for r in replies]
candidates["keep"] = [j.get("keep") is True for j in judged]
candidates["llm_group"] = [j.get("group") for j in judged]
candidates["reason"] = [j.get("reason") for j in judged]
candidates.to_csv(OUT / "seed_candidates.csv", index=False)

In [28]:
# "other" never counts as kept, even if the LLM said keep
kept = candidates[candidates["keep"] & candidates["llm_group"].isin(GROUPS)]
print(len(kept), "of", len(candidates), "kept")

94 of 160 kept


In [29]:
# prefilter group vs LLM group
pd.crosstab(kept["group"], kept["llm_group"])

llm_group,animal,one_person,two_people,vehicle
group,,,,
animal,16,3,2,0
one_person,0,22,3,0
two_people,0,0,25,0
vehicle,0,0,0,23


the prefiler and LLM mostly agree

Lets review some rejected captions

In [30]:
candidates.loc[~candidates["keep"], ["caption", "group", "llm_group", "reason"]]

,caption,group,llm_group,reason
3,a cartoon monkey is jumping on a pirate ship,animal,other,"cartoon, not a real agent"
4,a female fashion show females are walking in and out the lane walkway,animal,other,"multiple actions, scene change"
7,two male seeing their good friend in heaven as they all receives their halo,animal,other,"fictional characters, not real agents"
9,a black cat sits in a girl lap and playing,animal,animal,static state
13,a cartoon animals singing in the clouds,animal,other,"cartoon, not a real agent"
...,...,...,...,...
148,a commercial airliner is crashing inside the plane drama unfolds,vehicle,other,"fictional scenario, not real"
151,the trucks carry away huge tanks and the soldiers march on the road,vehicle,other,"multiple actions, scene change"
152,a car drives around the grass on a sunny day in this animation,vehicle,other,"animation, not a real agent"
153,different motorcycles and scooters all parked,vehicle,vehicle,static state


## Sampling the captions

We sample from the llm's agent group. We sample equally from each group.


In [2]:
def sample(group, n):
    pool = kept[kept["llm_group"] == group]
    if len(pool) < n:
        print(f"only {len(pool)} kept for {group}; raise CANDIDATES_PER_GROUP")
    return pool.sample(n=min(n, len(pool)), random_state=SEED)

In [33]:
per_group = [sample(group, n) for group, n in SEED_QUOTA.items()]
seeds = pd.concat(per_group, ignore_index=True)

# keep the LLM's group: it is the one we sampled by
seeds = seeds[["video_id", "sen_id", "llm_group", "caption"]]
seeds = seeds.rename(columns={"llm_group": "group"})

seeds.to_csv(OUT / "seeds.csv", index=False)
seeds

,video_id,sen_id,group,caption
0,video6799,27429,two_people,a man is singing a song and another man is playing drums in a stage
1,video6763,149746,two_people,two young men sit at a cafeteria table and talk about science fiction
2,video4106,170724,one_person,a male is demonstrating how he applies makeup with several items
3,video9372,86100,one_person,a man barbequeing some patties on a grill
4,video9253,164666,animal,a bird flies over the sea while rocky cliffs are shown in the background
5,video2046,77170,animal,fish are swimming and so is an octopus
6,video2214,195487,animal,a bird is flying and sea water is showed
7,video6978,101352,animal,sea creatures move along the ocean bottom
8,video489,195445,vehicle,boats bounce along on small waves near a high rocky coast line
9,video1725,144160,vehicle,a bus pulls up to a curb then pulls off


Now we can unload the LLM from GPU memory

In [34]:
import gc

del llm, tokenizer
gc.collect()
torch.cuda.empty_cache()

print("VRAM allocated: %.1f GB | reserved: %.1f GB" % (torch.cuda.memory_allocated() / 1e9, torch.cuda.memory_reserved() / 1e9))

VRAM allocated: 0.0 GB | reserved: 0.1 GB


## Video generation
We generate clips with Wan2.1-T2V-1.3B. Each seed caption gets a few clips with fixed random seeds, so every clip can be reproduced.

In [3]:
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

T2V_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

We will load the text encoder, generate the caption embeddings, and then unload the text encoder  in order to free up GPU memory for the video generator.

In [4]:
import gc
from transformers import UMT5EncoderModel

seeds = pd.read_csv(OUT / "seeds.csv")
NEGATIVE_PROMPT = "low quality, worst quality, blurry, jpeg artifacts, distorted, deformed, watermark, subtitles, text"

In [37]:
text_encoder = UMT5EncoderModel.from_pretrained(
    T2V_ID, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map="cuda"
)
text_pipe = WanPipeline.from_pretrained(
    T2V_ID, text_encoder=text_encoder, transformer=None, vae=None, torch_dtype=torch.bfloat16
)

Loading weights: 100%|██████████| 242/242 [00:09<00:00, 24.55it/s]
/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...: 100%|██████████| 3/3 [00:02<00:00,  1.40it/s]


In [38]:
embeds = {}
with torch.no_grad():
    for caption in seeds["caption"]:
        pos, neg = text_pipe.encode_prompt(prompt=caption, negative_prompt=NEGATIVE_PROMPT, device="cuda")
        embeds[caption] = (pos.cpu(), neg.cpu())

torch.save(embeds, OUT / "caption_embeds.pt")

In [39]:
del text_pipe, text_encoder; gc.collect(); torch.cuda.empty_cache()
print(len(embeds), "captions encoded | VRAM now %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

12 captions encoded | VRAM now 0.0 GB


In [6]:
embeds = torch.load(OUT / "caption_embeds.pt")

assert set(seeds["caption"]) <= set(embeds), "some captions have no embeddings: rerun the text encoding cells"

Now we load the video generator

In [5]:
vae = AutoencoderKLWan.from_pretrained(T2V_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(T2V_ID, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=torch.bfloat16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.to("cuda")

/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/home/skiiyuru/unitn/trends-and-apps-cv/open-temporal-bias/.venv/lib/python3.12/site-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
Loading pipeline components...: 100%|██████████| 3/3 [00:02<00:00,  1.03it/s]


WanPipeline {
  "_class_name": "WanPipeline",
  "_diffusers_version": "0.43.0",
  "_name_or_path": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
  "boundary_ratio": null,
  "expand_timesteps": false,
  "scheduler": [
    "diffusers",
    "UniPCMultistepScheduler"
  ],
  "text_encoder": [
    null,
    null
  ],
  "tokenizer": [
    null,
    null
  ],
  "transformer": [
    "diffusers",
    "WanTransformer3DModel"
  ],
  "transformer_2": [
    null,
    null
  ],
  "vae": [
    "diffusers",
    "AutoencoderKLWan"
  ]
}

Wan2.1 defaults for 480p: 832x480, 81 frames at 16 fps (~5 s), guidance 5.0, 50 steps.

In [7]:
import time

DEFAULTS = dict(height=480, width=832, num_frames=81, guidance_scale=5.0, num_inference_steps=50)
FPS = 16
CLIPS_PER_CAPTION = 2 
CLIP_SEEDS = list(range(CLIPS_PER_CAPTION))
pipe.vae.enable_tiling() # decode in tiles: caps VAE memory for full-length clips

VIDEO_DIR = OUT / "videos"; VIDEO_DIR.mkdir(exist_ok=True)
print(len(seeds), "captions x", CLIPS_PER_CAPTION, "clips =", len(seeds) * CLIPS_PER_CAPTION, "clips")

12 captions x 2 clips = 24 clips


In [8]:
def generate(caption, seed, **overrides):
    """One clip from the saved caption embeddings. The noise comes from a CPU generator, so seeds match across machines."""
    
    pos, neg = embeds[caption]
    return pipe(prompt_embeds=pos.to("cuda"), negative_prompt_embeds=neg.to("cuda"),
                **{**DEFAULTS, **overrides},
                generator=torch.Generator("cpu").manual_seed(seed)).frames[0]


Let's generate one short clip (33 frames, 20 steps). Check the time and peak VRAM before the full run.

In [15]:
torch.cuda.reset_peak_memory_stats()
t = time.time()
frames = generate(seeds["caption"][0], seed=0, num_frames=33, num_inference_steps=20)
export_to_video(frames, str(VIDEO_DIR / "smoke_test.mp4"), fps=FPS)
print("%.0f s | peak VRAM %.1f GB" % (time.time() - t, torch.cuda.max_memory_allocated() / 1e9))

100%|██████████| 20/20 [00:51<00:00,  2.59s/it]


65 s | peak VRAM 5.0 GB


In [16]:
from IPython.display import Video

Video(str(VIDEO_DIR / "smoke_test.mp4"), embed=True, width=480)

In [12]:
seeds["caption"][0]

'a man is singing a song and another man is playing drums in a stage'

In [ ]:
# lets try a full clip (81 frames, 50 steps) to see how long it takes 
torch.cuda.reset_peak_memory_stats()
t = time.time()
frames = generate(seeds["caption"][0], seed=0)
export_to_video(frames, str(VIDEO_DIR / f"{seeds['video_id'][0]}_seed0.mp4"), fps=FPS)
print("%.0f s | peak VRAM %.1f GB" % (time.time() - t, torch.cuda.max_memory_allocated() / 1e9))

In [14]:
Video(str(VIDEO_DIR / f"{seeds['video_id'][0]}_seed0.mp4"), embed=True, width=480)

### Generating all the samples

For now each sample batch = 12 captions. We generate 2 clips per caption, each with a different random seed.


In [18]:
records = []
for row in seeds.itertuples():
    for s in CLIP_SEEDS:
        path = VIDEO_DIR / f"{row.video_id}_seed{s}.mp4"
        if not path.exists():
            t = time.time()
            export_to_video(generate(row.caption, s), str(path), fps=FPS)
            print(f"{path.name}: {time.time() - t:.0f} s | {row.caption}")
        records.append({"video_id": row.video_id, "group": row.group, "caption": row.caption,
                        "seed": s, "path": str(path), "negative_prompt": NEGATIVE_PROMPT, **DEFAULTS})

clips = pd.DataFrame(records)
clips.to_csv(OUT / "clips.csv", index=False)
clips

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [07:18<00:00,  8.78s/it]


video6799_seed1.mp4: 465 s | a man is singing a song and another man is playing drums in a stage


100%|██████████| 50/50 [07:19<00:00,  8.80s/it]


video6763_seed0.mp4: 464 s | two young men sit at a cafeteria table and talk about science fiction


100%|██████████| 50/50 [07:19<00:00,  8.78s/it]


... (38 outputs left out)

video5150_seed0.mp4: 465 s | a car comes to a screeching halt on an open road


100%|██████████| 50/50 [07:21<00:00,  8.82s/it]


video5150_seed1.mp4: 468 s | a car comes to a screeching halt on an open road


,video_id,group,caption,seed,path,negative_prompt,height,width,num_frames,guidance_scale,num_inference_steps
0,video6799,two_people,a man is singing a song and another man is pla...,0,outputs/videos/video6799_seed0.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
1,video6799,two_people,a man is singing a song and another man is pla...,1,outputs/videos/video6799_seed1.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
2,video6763,two_people,two young men sit at a cafeteria table and tal...,0,outputs/videos/video6763_seed0.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
3,video6763,two_people,two young men sit at a cafeteria table and tal...,1,outputs/videos/video6763_seed1.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
4,video4106,one_person,a male is demonstrating how he applies makeup ...,0,outputs/videos/video4106_seed0.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
5,video4106,one_person,a male is demonstrating how he applies makeup ...,1,outputs/videos/video4106_seed1.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
6,video9372,one_person,a man barbequeing some patties on a grill,0,outputs/videos/video9372_seed0.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
7,video9372,one_person,a man barbequeing some patties on a grill,1,outputs/videos/video9372_seed1.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
8,video9253,animal,a bird flies over the sea while rocky cliffs a...,0,outputs/videos/video9253_seed0.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50
9,video9253,animal,a bird flies over the sea while rocky cliffs a...,1,outputs/videos/video9253_seed1.mp4,"low quality, worst quality, blurry, jpeg artif...",480,832,81,5.0,50


In [ ]:
import base64
from IPython.display import HTML

def show(rows, width=360):
    """Clips side by side, with their caption and seed. Videos are embedded, since VS Code can't load local files in outputs."""
    
    html = "".join(
        f'<div style="display:inline-block;margin:4px;width:{width}px;vertical-align:top">'
        f'<video src="data:video/mp4;base64,{base64.b64encode(open(r.path, "rb").read()).decode()}" '
        f'width="{width}" controls loop muted></video>'
        f'<div style="font-size:16px">{r.caption} (seed {r.seed})</div></div>'
        for r in rows.itertuples())
    return HTML(html)

show(clips[clips["group"] == VEHICLE])